# Compound Comparator Demo

Sometimes fields only make sense together. Scoring them independently gives wrong results.

This notebook demonstrates:
1. **The problem** -- independent scoring gives misleading partial credit when part of a full name is wrong
2. **The solution** -- a compound batch comparator that scores `surname` + `name` as one full name
3. **Group by parent** -- when the same fields appear in multiple places (student vs teacher), the comparator correctly pairs each person's fields
4. **Primary + supporting** -- one field gets the score, the other is skipped (1 field counted, not 2)

## Data

Each record has a **student** and a **teacher**, both with `surname` + `name`.

The compound comparator receives all 4 name fields per record and must group them correctly:
`student.surname` pairs with `student.name`, not with `teacher.name`.

Two records:
1. **Student name wrong**, teacher correct
2. **Teacher surname wrong**, student correct

In [14]:
GOLD = [
    {
        "student": {"surname": "Smith", "name": "John"},
        "teacher": {"surname": "Chen", "name": "Wei"},
    },
    {
        "student": {"surname": "Kim", "name": "Soo"},
        "teacher": {"surname": "Mueller", "name": "Anna"},
    },
]

EXTRACTED = [
    {   # student name wrong, teacher correct
        "student": {"surname": "Smith", "name": "Jane"},
        "teacher": {"surname": "Chen", "name": "Wei"},
    },
    {   # student correct, teacher surname wrong
        "student": {"surname": "Kim", "name": "Soo"},
        "teacher": {"surname": "Schmidt", "name": "Anna"},
    },
]

## Step 1: Independent scoring (the problem)

Each field scored on its own. Record 0: family_name="Smith" matches, given_name="Jane" vs "John" mismatches.

The extractor gets credit for "Smith" even though it identified a completely different person.

In [15]:
from struct_extract_eval.evaluator import evaluate

# Schema with independent exact comparators
NAME_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "exact"},
    "name": {"type": "string", "x-eval-compare": "exact"},
}
independent_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(NAME_FIELDS)},
        "teacher": {"type": "object", "properties": dict(NAME_FIELDS)},
    },
}

result_independent = evaluate(GOLD, EXTRACTED, schema=independent_schema)

print("Independent scoring (exact per field):")
print(f"  Mean F1: {result_independent.mean_f1:.3f}")
print()
for record in result_independent.records:
    print(f"  Record {record.record_id} -- F1: {record.f1:.3f}, Precision: {record.precision:.3f}, Recall: {record.recall:.3f}")
    for fr in record.field_results:
        print(f"    {fr.path:<30} {fr.score:.1f}  {fr.status}")
    print()

Independent scoring (exact per field):
  Mean F1: 0.750

  Record 0 -- F1: 0.750, Precision: 0.750, Recall: 0.750
    student.surname                1.0  match
    student.name                   0.0  mismatch
    teacher.surname                1.0  match
    teacher.name                   1.0  match

  Record 1 -- F1: 0.750, Precision: 0.750, Recall: 0.750
    student.surname                1.0  match
    student.name                   1.0  match
    teacher.surname                0.0  mismatch
    teacher.name                   1.0  match



**Record 0:** `student.surname`="Smith" matches, but `student.name`="Jane" vs "John" mismatches.
Independent scoring gives credit for "Smith" -- but "Smith, Jane" is not "Smith, John".

**Record 1:** `teacher.name`="Anna" matches, but `teacher.surname`="Schmidt" vs "Mueller" mismatches.
Same problem -- credit for "Anna" when "Schmidt, Anna" is a different person from "Mueller, Anna".

## Step 2: Define a compound comparator

A `NameCompoundComparator` that:
- Groups `family_name` + `given_name` by parent path
- Concatenates them into a full name, normalizes, compares
- Handles swapped order
- Primary field (`family_name`) gets the score
- Supporting field (`given_name`) gets `skip=True` (excluded from metrics) so that this compound counts as 1 field, not 2, and take the family name as the primary field for reporting purposes.

In [22]:
from struct_extract_eval.core.comparators.comparator import BatchItem, ComparatorResult


class NameCompoundComparator:
    """Compound comparator: scores surname + name as one full name.

    Groups items by parent path (e.g. "student" vs "teacher") so each
    person's name is evaluated independently.

    Primary field (surname) gets the compound score.
    Supporting field (name) gets skip=True (excluded from metrics).
    """

    is_batch = True

    def __init__(self, primary_field: str = "surname") -> None:
        self.primary_field = primary_field

    def __call__(self, items: list[BatchItem]) -> list[ComparatorResult | None]:
        # Group by parent path: "student.surname" -> parent "student" or "teacher.surname" -> parent "teacher"
        by_parent: dict[str, list[tuple[int, BatchItem]]] = {}
        for i, item in enumerate(items):
            parent = item.path.rsplit(".", 1)[0] if "." in item.path else ""
            by_parent.setdefault(parent, []).append((i, item))

        result_by_index: dict[int, ComparatorResult] = {}
        for parent, group in by_parent.items():
            fields: dict[str, tuple[int, BatchItem]] = {}
            for idx, item in group:
                field_name = item.path.rsplit(".", 1)[-1] if "." in item.path else item.path
                fields[field_name] = (idx, item)

            surname_entry = fields.get("surname")
            name_entry = fields.get("name")

            if surname_entry is None or name_entry is None:
                for idx, _ in group:
                    result_by_index[idx] = ComparatorResult(
                        score=0.0, comparator="name_compound",
                        reason="incomplete compound",
                    )
                continue

            score = self._compare(
                str(surname_entry[1].gold_compared or ""),
                str(name_entry[1].gold_compared or ""),
                str(surname_entry[1].extracted_compared or ""),
                str(name_entry[1].extracted_compared or ""),
            )

            for field_name, (idx, _) in fields.items():
                if field_name == self.primary_field:
                    result_by_index[idx] = ComparatorResult(
                        score=score, comparator="name_compound",
                        reason="compound: full name matches" if score == 1.0 else "compound: full name mismatches",
                    )
                else:
                    result_by_index[idx] = ComparatorResult(
                        score=0.0, comparator="name_compound",
                        reason=f"compound with {parent}.{self.primary_field}",
                        skip=True,
                    )

        return [result_by_index.get(i) for i in range(len(items))]

    @staticmethod
    def _compare(gold_surname: str, gold_name: str, ext_surname: str, ext_name: str) -> float:
        gold_full = f"{gold_name} {gold_surname}"
        ext_full = f"{ext_name} {ext_surname}"
        return 1.0 if gold_full == ext_full else 0.0


print("NameCompoundComparator defined.")

NameCompoundComparator defined.


## Step 3: Register and run with compound comparator

In [23]:
from struct_extract_eval.core.comparators.registry import _registry, register

# Safe for re-runs
_registry.pop("name_compound", None)
register("name_compound", NameCompoundComparator())

# Schema: both student and teacher use compound for name fields
COMPOUND_NAME_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "name_compound"},
    "name": {"type": "string", "x-eval-compare": "name_compound"},
}
compound_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(COMPOUND_NAME_FIELDS)},
        "teacher": {"type": "object", "properties": dict(COMPOUND_NAME_FIELDS)},
    },
}

result_compound = evaluate(GOLD, EXTRACTED, schema=compound_schema)

print("Compound scoring (surname + name as one full name, grouped by parent):")
print(f"  Mean F1: {result_compound.mean_f1:.3f}")
print()
for record in result_compound.records:
    print(f"  Record {record.record_id} -- F1: {record.f1:.3f}, Precision: {record.precision:.3f}, Recall: {record.recall:.3f}")
    for fr in record.field_results:
        reason = f"  ({fr.reason})" if fr.reason else ""
        print(f"    {fr.path:<30} {fr.score:.1f}  {fr.status:<12}{reason}")
    print()

Compound scoring (surname + name as one full name, grouped by parent):
  Mean F1: 0.500

  Record 0 -- F1: 0.500, Precision: 0.500, Recall: 0.500
    student.surname                0.0  mismatch      (compound: full name mismatches)
    student.name                   0.0  skipped       (compound with student.surname)
    teacher.surname                1.0  match         (compound: full name matches)
    teacher.name                   0.0  skipped       (compound with teacher.surname)

  Record 1 -- F1: 0.500, Precision: 0.500, Recall: 0.500
    student.surname                1.0  match         (compound: full name matches)
    student.name                   0.0  skipped       (compound with student.surname)
    teacher.surname                0.0  mismatch      (compound: full name mismatches)
    teacher.name                   0.0  skipped       (compound with teacher.surname)



### What changed

- **Record 0 (student name wrong):** Independent gives `student.surname`=1.0 (misleading).
  Compound gives `student.surname`=0.0 (correct -- full name is wrong). Teacher is correct
  in both approaches.
- **Record 1 (teacher surname wrong):** Independent gives `teacher.name`=1.0 (misleading).
  Compound gives `teacher.surname`=0.0 (correct). Student is correct in both.
- **Total fields** dropped because `name` is skipped (supporting field) in both student and teacher.

### Group by parent -- why it matters

The comparator receives 4 pending items per record, for example, in record 0:

```
process_batches sends all "name_compound" fields:
  student.surname    gold="Smith"      extracted="Smith"
  student.name       gold="John"       extracted="Jane"
  teacher.surname    gold="Chen"       extracted="Chen"
  teacher.name       gold="Wei"        extracted="Wei"
```

The handler groups by parent path:

```
  "student": [student.surname, student.name]   -> "John Smith" vs "Jane Smith" -> 0
  "teacher": [teacher.surname, teacher.name]   -> "Wei Chen" vs "Wei Chen" -> 1
```

Without grouping, the handler couldn't tell which surname belongs to which name --
it would see 4 items with no way to pair them correctly.

### The flow

```
score_record():
  student.surname -> pending (name_compound)
  student.name    -> pending (name_compound)
  teacher.surname -> pending (name_compound)
  teacher.name    -> pending (name_compound)

process_batches():
  group all 4 -> call NameCompoundComparator once
  handler groups by parent internally:
    "student": full name mismatches -> surname=mismatch, name=skipped
    "teacher": full name matches -> surname=match, name=skipped

Metrics: 2 fields per record (student.surname + teacher.surname), not 4.
```

## Step 4: Compound inside arrays

What if students are in an array? The same `surname` + `name` fields appear for every element,
all with the same schema path `students[].surname`.

This works because the scorer writes **instance paths** (`students[0].surname`, `students[1].surname`).
When the compound handler groups by parent, each element gets its own group:

```
students[0].surname + students[0].name  -> one full name
students[1].surname + students[1].name  -> another full name
```

Without instance paths, all items would have parent `students[]` and the handler couldn't
tell which surname belongs to which name.

In [24]:
GOLD_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "John"},
        {"surname": "Kim", "name": "Soo"},
    ]},
]

EXTRACTED_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "Jane"},   # name wrong
        {"surname": "Kim", "name": "Soo"},       # correct
    ]},
]

array_schema = {
    "type": "object",
    "properties": {
        "students": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "surname": {"type": "string", "x-eval-compare": "name_compound"},
                    "name": {"type": "string", "x-eval-compare": "name_compound"},
                },
            },
        },
    },
}

result_array = evaluate(GOLD_ARRAY, EXTRACTED_ARRAY, schema=array_schema)

print("Compound inside array:")
print(f"  Mean F1: {result_array.mean_f1:.3f}")
print()
for record in result_array.records:
    print(f"  Record {record.record_id} -- F1: {record.f1:.3f}")
    for fr in record.field_results:
        reason = f"  ({fr.reason})" if fr.reason else ""
        print(f"    {fr.path:<30} {fr.score:.1f}  {fr.status:<12}{reason}")
    print()

print("Instance paths with array index ensure correct grouping:")
print("  students[0].surname + students[0].name  ->  parent 'students[0]'  ->  one full name")
print("  students[1].surname + students[1].name  ->  parent 'students[1]'  ->  another full name")

Compound inside array:
  Mean F1: 0.500

  Record 0 -- F1: 0.500
    students[0].surname            0.0  mismatch      (compound: full name mismatches)
    students[0].name               0.0  skipped       (compound with students[0].surname)
    students[1].surname            1.0  match         (compound: full name matches)
    students[1].name               0.0  skipped       (compound with students[1].surname)

Instance paths with array index ensure correct grouping:
  students[0].surname + students[0].name  ->  parent 'students[0]'  ->  one full name
  students[1].surname + students[1].name  ->  parent 'students[1]'  ->  another full name
